# Python to SQL, and back again
In this codealong we will show you how to create a relational database from your pandas DataFrames.
> **To run this notebook you will need to work locally and not on colab.**

---
## 1.&nbsp; Import libraries 💾
If you haven't already installed sqlalchemy, you will need to. Uncomment the code below, install, and then recomment the code - you only need to install it once.

In [ ]:
%pip install sqlalchemy pymysql

In [20]:
import pandas as pd


---
## 2.&nbsp; Relational Databases 📂

Creating DataFrames in python and pandas often results in tables with repeated information, as shown in the example below.
<br>

| author_name | book_title | year_published |
| --- | --- | --- |
| Arthur Conan Doyle | The Adventures of Sherlock Holmes | 1887 |
| J.R.R. Tolkien | The Hobbit | 1937 |
| J.R.R. Tolkien | The Lord of the Rings | 1954 |
| Harper Lee | To Kill a Mockingbird | 1960 |
| Harper Lee | Go Set a Watchman | 2015 |
<br>

This can be problematic for relational databases, which are designed to store data efficiently and avoid redundancy. To address this issue, we will separate the author and book information into two tables: authors and books. This approach eliminates duplicate data, ensuring data integrity and optimising storage.
<br>

| author_id | author_name |
| --- | --- |
| 1 | Arthur Conan Doyle |
| 2 | J.R.R. Tolkien |
| 3 | Harper Lee |
<br>

| book_id | book_title | year_published | author_id |
|---|---|---|---|
| 1 | The Adventures of Sherlock Holmes | 1887 | 1 |
| 2 | The Hobbit | 1937 | 2 |
| 3 | The Lord of the Rings | 1954 | 2 |
| 4 | To Kill a Mockingbird | 1960 | 3 |
| 5 | Go Set a Watchman | 2015 | 3 |

---
## 3.&nbsp; Creating the authors table with python 🐍
Let's start by creating the original DataFrame, including the repeated data.

In [21]:
names = ["Arthur Conan Doyle", "J.R.R. Tolkien", "J.R.R. Tolkien", "Harper Lee", "Harper Lee"]
titles = ["The Adventures of Sherlock Holmes", "The Hobbit", "The Lord of the Rings", "To Kill a Mockingbird", "Go Set a Watchman"]
years = [1887, 1937, 1954, 1960, 2015]

non_relational_df = pd.DataFrame({"author_name": names,
                                  "book_title": titles,
                                  "year_published": years})

non_relational_df

,author_name,book_title,year_published
0,Arthur Conan Doyle,The Adventures of Sherlock Holmes,1887
1,J.R.R. Tolkien,The Hobbit,1937
2,J.R.R. Tolkien,The Lord of the Rings,1954
3,Harper Lee,To Kill a Mockingbird,1960
4,Harper Lee,Go Set a Watchman,2015


Now, let's select only the authors without any duplicates.

In [22]:
authors_unique = non_relational_df["author_name"].unique()

authors_df = pd.DataFrame({"author_name": authors_unique})

authors_df

,author_name
0,Arthur Conan Doyle
1,J.R.R. Tolkien
2,Harper Lee


Fantastic! This DataFrame will be the foundation of our authors table.

---
## 4.&nbsp; Creating the matching authors table with SQL 💻

Ok, now we're ready to store this DataFrame in SQL. Before we can send the information in SQL, we need to make a table that has the same columns and data types to recieve the data. While we are creating a table for authors, we can also create the books table too.

Open MySQL Workbench, open a local connection, and open a new file. Then copy and paste the code from below.

```sql
-- Drop the database if it already exists
DROP DATABASE IF EXISTS sql_workshop ;

-- Create the database
CREATE DATABASE sql_workshop;

-- Use the database
USE sql_workshop;

-- Create the 'authors' table
CREATE TABLE authors (
    author_id INT AUTO_INCREMENT, -- Automatically generated ID for each author
    author_name VARCHAR(255) NOT NULL, -- Name of the author
    PRIMARY KEY (author_id) -- Primary key to uniquely identify each author
);

-- Create the 'books' table
CREATE TABLE books (
    book_id INT AUTO_INCREMENT, -- Automatically generated ID for each book
    book_title VARCHAR(255) NOT NULL, -- Title of the book
    year_published INT, -- Year the book was published
    author_id INT, -- ID of the author who wrote the book
    PRIMARY KEY (book_id), -- Primary key to uniquely identify each book
    FOREIGN KEY (author_id) REFERENCES authors(author_id) -- Foreign key to connect each book to its author
);
```

To download the sql file that we will follow for this section, [click here](https://drive.google.com/uc?export=download&id=1tln_33FM7D9wLckzxacBJNcMYqtyxybE)

If you'd like more information about MySQL data types [click here](https://www.w3schools.com/mysql/mysql_datatypes.asp).

---
## 5.&nbsp; Sending the information from this notebook to sql 📠
To establish a connection with the SQL database, we need to provide the notebook with the necessary information, which we do using the connection string below. You will need to modify only the password variable, which should match the password you set during MySQL Workbench installation.

In [23]:
from getpass import getpass

schema = "sql_workshop"
host = "127.0.0.1"
user = "root"
password = getpass("Enter MySQL password: ")
port = 3306

connection_string = (
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"
)

To send information to our sql databse we use the pandas method `.to_sql()`. The argument `if_exists="append"` says that we don't want to overwrite any existing data, but add on to what is already there.

In [24]:
authors_df.to_sql('authors',
                  if_exists='append',
                  con=connection_string,
                  index=False)

3

Now, have a look at the table `authors` in MySQL Workbench, you should see that the names of the authors have appeared.

---
## 6.&nbsp; Retrieving information from sql to this notebook 📥
It's not only possible to send information to a SQL database, but also retrieve it too. Using `.read_sql()` in combination with the `connection_string` we can access the required data.

In [25]:
authors_from_sql = pd.read_sql("authors", con=connection_string)
authors_from_sql

,author_id,author_name
0,1,Arthur Conan Doyle
1,2,J.R.R. Tolkien
2,3,Harper Lee
3,4,Arthur Conan Doyle
4,5,J.R.R. Tolkien
5,6,Harper Lee


Using this same method, we can also perform SQL queries to only bring back certain sections of information instead of the whole DataFrame.

In [26]:
pd.read_sql("""
            SELECT DISTINCT author_name
            FROM authors
            """,
            con=connection_string)

,author_name
0,Arthur Conan Doyle
1,J.R.R. Tolkien
2,Harper Lee


---
## 7.&nbsp; Preparing and sending the books table 📚
By extracting the authors table from our SQL database, we gain access to the unique identifier `author_id` assigned to each author. These `author_id`'s serve as pointers to their corresponding author records, allowing us to seamlessly link the `author_id`'s in the books table to their respective authors in the authors table, thereby completing the books table.

In [27]:
books_df = non_relational_df.merge(authors_from_sql,
                                   on = "author_name",
                                   how="left")

books_df

,author_name,book_title,year_published,author_id
0,Arthur Conan Doyle,The Adventures of Sherlock Holmes,1887,1
1,Arthur Conan Doyle,The Adventures of Sherlock Holmes,1887,4
2,J.R.R. Tolkien,The Hobbit,1937,2
3,J.R.R. Tolkien,The Hobbit,1937,5
4,J.R.R. Tolkien,The Lord of the Rings,1954,2
5,J.R.R. Tolkien,The Lord of the Rings,1954,5
6,Harper Lee,To Kill a Mockingbird,1960,3
7,Harper Lee,To Kill a Mockingbird,1960,6
8,Harper Lee,Go Set a Watchman,2015,3
9,Harper Lee,Go Set a Watchman,2015,6


In [28]:
books_df = books_df.drop(columns=["author_name"])

books_df

,book_title,year_published,author_id
0,The Adventures of Sherlock Holmes,1887,1
1,The Adventures of Sherlock Holmes,1887,4
2,The Hobbit,1937,2
3,The Hobbit,1937,5
4,The Lord of the Rings,1954,2
5,The Lord of the Rings,1954,5
6,To Kill a Mockingbird,1960,3
7,To Kill a Mockingbird,1960,6
8,Go Set a Watchman,2015,3
9,Go Set a Watchman,2015,6


In [29]:
books_df.to_sql('books',
                if_exists='append',
                con=connection_string,
                index=False)

10

In [30]:
books_from_sql = pd.read_sql("books", con=connection_string)
books_from_sql

,book_id,book_title,year_published,author_id
0,1,The Adventures of Sherlock Holmes,1887,1
1,2,The Hobbit,1937,2
2,3,The Lord of the Rings,1954,2
3,4,To Kill a Mockingbird,1960,3
4,5,Go Set a Watchman,2015,3
5,6,The Adventures of Sherlock Holmes,1887,1
6,7,The Adventures of Sherlock Holmes,1887,4
7,8,The Hobbit,1937,2
8,9,The Hobbit,1937,5
9,10,The Lord of the Rings,1954,2


---
## 8.&nbsp; Challenge 😃
Now that you've learnt how to send and retrieve information, it's your turn to show off your skills. Create multiple tables in SQL for the data you scrapped about cities from Wikipedia. One should just be a table about the cities, the others should be facts about the cities.

| city_id | city |
| --- | --- |
| 1 | Berlin |
| 2 | Hamburg |
| 3 | Munich |

<br>

| City ID | Population | Year Data Retrieved |
|---|---|---|
| 1 | 3,850,809 | 2024 |
| 2 | 1,945,532 | 2024 |
| 3 | 1,512,491 | 2024 |

> **Pro Tip:** Visualise your relational database with pen and paper before you start coding. This can help you to identify any potential problems or inconsistencies in your design, and it can also make the coding process more efficient.

In [31]:
cities_df = pd.read_csv("cities.csv")
population_df = pd.read_csv("populations.csv")

display(cities_df)
display(population_df)

,city_name,country,latitude,longitude
0,Berlin,Germany,52.5200,13.405
1,Hamburg,Germany,53.5500,10.000
2,Munich,Germany,48.1375,11.575


,city_name,population,timestamp
0,Berlin,"3,596,999",2026-07-20 15:58:44.547178
1,Hamburg,"1,973,896",2026-07-20 15:58:57.709114
2,Munich,"1,505,005",2026-07-20 15:58:58.644991


In [32]:
gans_schema = "gans"

gans_connection_string = (
    f"mysql+pymysql://{user}:{password}"
    f"@{host}:{port}/{gans_schema}"
)

In [ ]:
cities_df.to_sql(
    "cities",
    con=gans_connection_string,
    if_exists="append",
    index=False
)

In [ ]:
cities_from_sql = pd.read_sql(
    "cities",
    con=gans_connection_string
)

cities_from_sql

In [ ]:
# Adding city_id to the population data

In [ ]:
population_to_sql = population_df.merge(
    cities_from_sql[["city_id", "city_name"]],
    on="city_name",
    how="left"
)

population_to_sql

In [ ]:
# Cleaning the population and timestamp columns

In [ ]:
population_to_sql["population"] = (
    population_to_sql["population"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype(int)
)

population_to_sql["timestamp"] = pd.to_datetime(
    population_to_sql["timestamp"]
)

population_to_sql = population_to_sql.rename(
    columns={
        "timestamp": "timestamp_population"
    }
)

population_to_sql = population_to_sql[
    [
        "city_id",
        "population",
        "timestamp_population"
    ]
]

population_to_sql

In [34]:
# Uploading populations

In [ ]:
population_to_sql.to_sql(
    "populations",
    con=gans_connection_string,
    if_exists="append",
    index=False
)

In [26]:
# Verifying the final relational database

In [27]:
final_df = pd.read_sql(
    """
    SELECT
        c.city_id,
        c.city_name,
        c.country,
        c.latitude,
        c.longitude,
        p.population,
        p.timestamp_population
    FROM cities AS c
    JOIN populations AS p
        ON c.city_id = p.city_id
    ORDER BY c.city_id
    """,
    con=gans_connection_string
)

final_df

,city_id,city_name,country,latitude,longitude,population,timestamp_population
0,1,Berlin,Germany,52.5200,13.405,3596999,2026-07-20 15:58:45
1,2,Hamburg,Germany,53.5500,10.000,1973896,2026-07-20 15:58:58
2,3,Munich,Germany,48.1375,11.575,1505005,2026-07-20 15:58:59


In [ ]:
# defining my API key
from getpass import getpass

API_key = getpass("Enter OpenWeather API key: ")

In [3]:
import requests

params = {
    "lat": 52.52,
    "lon": 13.405,
    "appid": API_key,
    "units": "metric"
}

response = requests.get(
    "https://api.openweathermap.org/data/2.5/forecast",
    params=params,
    timeout=20
)

print(response.status_code)

200


In [ ]:
#Converting the response to a Python dictionary
weather_data = response.json()
weather_data

In [5]:
weather_data.keys()

dict_keys(['cod', 'message', 'cnt', 'list', 'city'])

In [6]:
weather_data["city"]

{'id': 6545310,
 'name': 'Mitte',
 'coord': {'lat': 52.52, 'lon': 13.405},
 'country': 'DE',
 'population': 329078,
 'timezone': 7200,
 'sunrise': 1784689835,
 'sunset': 1784747682}

In [7]:
# Selecting the first forecast from the list:
weather_data["list"][0]

{'dt': 1784732400,
 'main': {'temp': 19.02,
  'feels_like': 18.75,
  'temp_min': 19.02,
  'temp_max': 19.95,
  'pressure': 1016,
  'sea_level': 1016,
  'grnd_level': 1010,
  'humidity': 68,
  'temp_kf': -0.93,
  'dew_point': 12.99},
 'weather': [{'id': 500,
   'main': 'Rain',
   'description': 'light rain',
   'icon': '10d'}],
 'clouds': {'all': 99},
 'wind': {'speed': 5, 'deg': 257, 'gust': 9.07},
 'visibility': 10000,
 'pop': 0.2,
 'rain': {'3h': 0.12},
 'sys': {'pod': 'd'},
 'dt_txt': '2026-07-22 15:00:00'}

In [8]:
#extracting and printing useful information from that first forecast:
first_forecast = weather_data["list"][0]

print("Forecast time:", first_forecast["dt_txt"])
print("Temperature:", first_forecast["main"]["temp"])
print("Weather:", first_forecast["weather"][0]["description"])
print("Humidity:", first_forecast["main"]["humidity"])
print("Wind speed:", first_forecast["wind"]["speed"])
print("Rain probability:", first_forecast["pop"])

Forecast time: 2026-07-22 15:00:00
Temperature: 19.02
Weather: light rain
Humidity: 68
Wind speed: 5
Rain probability: 0.2


In [13]:
#creating weather forecast for Berlin:
# Creating weather forecasts for Berlin

weather_list = []

for forecast in weather_data["list"]:
    weather_list.append({
        "city_name": "Berlin",
        "forecast_time": forecast["dt_txt"],
        "temperature": forecast["main"]["temp"],
        "feels_like": forecast["main"]["feels_like"],
        "humidity": forecast["main"]["humidity"],
        "weather": forecast["weather"][0]["description"],
        "rain_probability": forecast["pop"],
        "rain_3h": forecast.get("rain", {}).get("3h", 0),
        "wind_speed": forecast["wind"]["speed"]
    })

In [14]:
# Converting the list into a pandas DataFrame:
import pandas as pd

berlin_weather_df = pd.DataFrame(weather_list)

berlin_weather_df.head()

,city_name,forecast_time,temperature,feels_like,humidity,weather,rain_probability,rain_3h,wind_speed
0,Berlin,2026-07-22 15:00:00,19.02,18.75,68,light rain,0.20,0.12,5.00
1,Berlin,2026-07-22 18:00:00,19.90,19.72,68,light rain,0.61,0.48,5.16
2,Berlin,2026-07-22 21:00:00,17.64,17.55,80,light rain,0.32,0.20,5.54
3,Berlin,2026-07-23 00:00:00,18.17,18.16,81,scattered clouds,0.00,0.00,5.88
4,Berlin,2026-07-23 03:00:00,16.38,16.40,89,broken clouds,0.00,0.00,5.96


In [17]:
# creating the reusable function
def get_weather(cities_df, API_key):
    all_weather = [] # creating an empty list

    for _, city in cities_df.iterrows():
        params = {
            "lat": city["latitude"],
            "lon": city["longitude"],
            "appid": API_key,
            "units": "metric"
        } # Uses each city’s latitude and longitude to request its forecast:

        response = requests.get(
            "https://api.openweathermap.org/data/2.5/forecast",
            params=params,
            timeout=20
        )

        response.raise_for_status()
        weather_data = response.json()

        for forecast in weather_data["list"]:
            all_weather.append({
                "city_name": city["city_name"],
                "forecast_time": forecast["dt_txt"],
                "temperature": forecast["main"]["temp"],
                "feels_like": forecast["main"]["feels_like"],
                "humidity": forecast["main"]["humidity"],
                "weather": forecast["weather"][0]["description"],
                "rain_probability": forecast["pop"],
                "rain_3h": forecast.get("rain", {}).get("3h", 0),
                "wind_speed": forecast["wind"]["speed"]
            })

    return pd.DataFrame(all_weather) #converting into dataframe

In [37]:
weather_df = get_weather(cities_df, API_key)

weather_df.head()

,city_name,forecast_time,temperature,feels_like,humidity,weather,rain_probability,rain_3h,wind_speed
0,Berlin,2026-07-22 15:00:00,19.26,18.99,67,light rain,0.20,0.12,5.00
1,Berlin,2026-07-22 18:00:00,20.02,19.83,67,light rain,0.61,0.48,5.16
2,Berlin,2026-07-22 21:00:00,17.64,17.55,80,light rain,0.32,0.20,5.54
3,Berlin,2026-07-23 00:00:00,18.17,18.16,81,scattered clouds,0.00,0.00,5.88
4,Berlin,2026-07-23 03:00:00,16.38,16.40,89,broken clouds,0.00,0.00,5.96


In [38]:
weather_df.groupby("city_name").size()

city_name
Berlin     40
Hamburg    40
Munich     40
dtype: int64

In [39]:
weather_df.shape

(120, 9)

In [40]:
weather_df["forecast_time"] = pd.to_datetime(
    weather_df["forecast_time"]
)

weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   city_name         120 non-null    object        
 1   forecast_time     120 non-null    datetime64[ns]
 2   temperature       120 non-null    float64       
 3   feels_like        120 non-null    float64       
 4   humidity          120 non-null    int64         
 5   weather           120 non-null    object        
 6   rain_probability  120 non-null    float64       
 7   rain_3h           120 non-null    float64       
 8   wind_speed        120 non-null    float64       
dtypes: datetime64[ns](1), float64(5), int64(1), object(2)
memory usage: 8.6+ KB


In [ ]:
## Weather API challenge conclusion

I used the OpenWeather 5 Day Forecast API to collect weather data for Berlin, Hamburg, and Munich.

The function uses each city's latitude and longitude and returns a clean pandas DataFrame. 
The resulting dataset contains 120 forecasts, including temperature, humidity, weather conditions, probability of rain, rainfall, and wind speed.

In [42]:
# to save in SQL i retrieve the city IDs:
cities_lookup = pd.read_sql(
    """
    SELECT city_id, city_name
    FROM cities
    """,
    con=gans_connection_string
)

cities_lookup

,city_id,city_name
0,1,Berlin
1,2,Hamburg
2,3,Munich


In [43]:
# Add city_id to the weather data
weather_to_sql = weather_df.merge(
    cities_lookup,
    on="city_name",
    how="left",
    validate="many_to_one"
)

In [44]:
# Add one retrieval timestamp for the whole API collection:
weather_to_sql["retrieved_at"] = (
    pd.Timestamp.now().floor("s")
)

In [45]:
# Removing city_name because MySQL can get it through city_id:
weather_to_sql = weather_to_sql[
    [
        "city_id",
        "forecast_time",
        "temperature",
        "feels_like",
        "humidity",
        "weather",
        "rain_probability",
        "rain_3h",
        "wind_speed",
        "retrieved_at"
    ]
]

weather_to_sql.head()

,city_id,forecast_time,temperature,feels_like,humidity,weather,rain_probability,rain_3h,wind_speed,retrieved_at
0,1,2026-07-22 15:00:00,19.26,18.99,67,light rain,0.20,0.12,5.00,2026-07-22 15:37:07
1,1,2026-07-22 18:00:00,20.02,19.83,67,light rain,0.61,0.48,5.16,2026-07-22 15:37:07
2,1,2026-07-22 21:00:00,17.64,17.55,80,light rain,0.32,0.20,5.54,2026-07-22 15:37:07
3,1,2026-07-23 00:00:00,18.17,18.16,81,scattered clouds,0.00,0.00,5.88,2026-07-22 15:37:07
4,1,2026-07-23 03:00:00,16.38,16.40,89,broken clouds,0.00,0.00,5.96,2026-07-22 15:37:07


In [49]:
#Confirm that no city IDs are missing:
weather_to_sql["city_id"].isna().sum()

np.int64(0)

In [57]:
# confirming that no citi ids are missing:
weather_to_sql["city_id"].isna().sum()

np.int64(0)

In [82]:
from getpass import getpass

RAPID_API_KEY = getpass("Enter RapidAPI key: ")

Enter RapidAPI key:  ········


In [ ]:
## Flight Arrivals API

import requests

url = (
    "https://aerodatabox.p.rapidapi.com/flights/airports/"
    "iata/BER/2026-07-24T00:00/2026-07-24T12:00"
)

querystring = {
    "withLeg": "true",
    "direction": "Arrival",
    "withCancelled": "true",
    "withCodeshared": "true",
    "withCargo": "false",
    "withPrivate": "false",
    "withLocation": "false"
}

headers = {
    "x-rapidapi-key": RAPID_API_KEY,
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
    "Content-Type": "application/json"
}

response = requests.get(
    url,
    headers=headers,
    params=querystring,
    timeout=20
)

print(response.status_code)

In [77]:
flight_data = response.json()

arrivals = flight_data["arrivals"]

print(len(arrivals))

146


In [90]:
airports_df = pd.DataFrame({
    "airport_id": [1, 2, 3],
    "airport_iata": ["BER", "HAM", "MUC"],
    "airport_name": [
        "Berlin Brandenburg Airport",
        "Hamburg Airport",
        "Munich Airport"
    ],
    "city_id": [1, 2, 3]
})

airports_df

,airport_id,airport_iata,airport_name,city_id
0,1,BER,Berlin Brandenburg Airport,1
1,2,HAM,Hamburg Airport,2
2,3,MUC,Munich Airport,3


In [91]:
airports_df.to_sql(
    "airports",
    con=gans_connection_string,
    if_exists="replace",
    index=False
)

3

In [78]:
# Collecting flights
import requests
import pandas as pd
from datetime import date, timedelta

def get_flights():

    tomorrow = date.today() + timedelta(days=1)

    url = (
        "https://aerodatabox.p.rapidapi.com/flights/airports/"
        f"iata/BER/{tomorrow}T00:00/{tomorrow}T12:00"
    )

    headers = {
        "x-rapidapi-key": RAPID_API_KEY,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com"
    }

    params = {
        "direction": "Arrival",
        "withLeg": "true"
    }

    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()

    rows = []

    for flight in response.json()["arrivals"]:
        rows.append({
            "city_id": 1,
            "flight_number": flight.get("number"),
            "origin_airport": flight.get(
                "departure", {}
            ).get("airport", {}).get("name"),
            "scheduled_arrival": flight.get(
                "arrival", {}
            ).get("scheduledTime", {}).get("local"),
            "flight_status": flight.get("status")
        })

    return pd.DataFrame(rows)

In [84]:
# creating a dataFrame
flights_df = get_flights()

flights_df.head()

,city_id,flight_number,origin_airport,scheduled_arrival,flight_status
0,1,TK 1220,Istanbul,2026-07-28 05:45+02:00,Expected
1,1,XQ 966,İzmir,2026-07-28 06:05+02:00,Expected
2,1,SR 1501,Beirut,2026-07-28 06:10+02:00,Expected
3,1,XQ 1766,Gaziantep,2026-07-28 06:45+02:00,Expected
4,1,W4 3109,Bucharest,2026-07-28 07:00+02:00,Expected


In [88]:
# Saving to MySQL
flights_df.to_sql(
    "flights",
    con=gans_connection_string,
    if_exists="replace",
    index=False
)

149